# 🧠 SwiGLU Gated Feed-Forward Network: Complete Hands-On Lab

Welcome! In this notebook, we master the <b>SwiGLU (Swish Gated Linear Unit)</b> architecture used in <b>Llama 3, DeepSeek, and Mirendil-Base</b> through hands-on paper-and-pencil exercises and runnable PyTorch code.

---

## 1. The Core Idea: Dynamic Gating

In standard MLPs (like GPT-2), every neuron is forced through an activation and passed directly to the next layer.
<br><br>
In <b>SwiGLU</b>, the input vector <b>x</b> splits into two parallel paths:
<ol>
  <li><b>Up-Projection (w_up):</b> The candidate knowledge features.</li>
  <li><b>Gate-Projection (w_gate):</b> A dynamic volume dial that decides how much of each feature to pass through.</li>
</ol>

<pre style='font-size: 16px; background: #161b22; color: #58a6ff; padding: 12px; border-radius: 8px;'>
SwiGLU(x) = ( SiLU(x · W_gate) ⊙ (x · W_up) ) · W_down
</pre>

*(Where <b>⊙</b> is simple element-wise multiplication)*.

---
## 2. Exercise 1: Computing SiLU (Swish) on Paper 📝

The gate uses the <b>SiLU</b> activation function:
<pre style='font-size: 16px; background: #161b22; color: #c9d1d9; padding: 12px; border-radius: 8px;'>
SiLU(z) = z · sigmoid(z) = z / (1 + e^(-z))
</pre>

### Test Case A: A Large Positive Signal (z = 4.0)
- When z = 4.0, sigmoid(4.0) ≈ 0.982
- <b>SiLU(4.0) = 4.0 · 0.982 ≈ 3.93</b>
- <i>Notice: For positive numbers, SiLU acts almost like a linear function! It lets strong signals pass straight through without diminishing.</i>

### Test Case B: A Negative Signal (z = -1.0)
- When z = -1.0, sigmoid(-1.0) ≈ 0.269
- <b>SiLU(-1.0) = -1.0 · 0.269 = -0.27</b>
- <i>Notice: Unlike ReLU (which hard-clips negative numbers to 0.0, killing neurons), SiLU allows a smooth negative gradient so neurons can recover during training!</i>

In [ ]:
import torch
import torch.nn.functional as F

z_pos = torch.tensor([4.0])
z_neg = torch.tensor([-1.0])

print(f"SiLU(4.0):   {F.silu(z_pos).item():.4f} (Nearly linear!)")
print(f"SiLU(-1.0):  {F.silu(z_neg).item():.4f} (Smooth negative valley!)")
print(f"ReLU(-1.0):  {F.relu(z_neg).item():.4f} (Hard zero - kills gradients)")

---
## 3. Exercise 2: The Element-Wise Gate Multiplication on Paper 📝

Suppose our candidate features vector is:
<pre style='font-size: 16px; background: #161b22; color: #7ee787; padding: 12px; border-radius: 8px;'>
candidate_features (w_up) = [ 10.0,  20.0,  -5.0 ]
</pre>

And our gate outputs:
<pre style='font-size: 16px; background: #161b22; color: #bc8cff; padding: 12px; border-radius: 8px;'>
gate_multiplier (SiLU(w_gate)) = [ 0.0,  1.0,  0.5 ]
</pre>

### Calculate the Element-Wise Product on Paper:
1. Coordinate 1: 10.0 · 0.0 = <b>0.0</b> &nbsp; (Silenced completely!)
2. Coordinate 2: 20.0 · 1.0 = <b>20.0</b> &nbsp; (Passed through at 100% strength)
3. Coordinate 3: -5.0 · 0.5 = <b>-2.5</b> &nbsp; (Scaled down by 50%)

<br>
<b>Gated Output Vector:</b><br>
<span style='color: #3fb950; font-size: 16px; font-weight: bold;'>
[ 0.0,  20.0,  -2.5 ]
</span>

In [ ]:
candidates = torch.tensor([10.0, 20.0, -5.0])
gate = torch.tensor([0.0, 1.0, 0.5])

gated_output = gate * candidates
print(f"Element-wise Gated Product: {gated_output.tolist()}")

---
## 4. Exercise 3: The 8/3 Scaling Rule Math 📝

In older models (like GPT-2):
- Standard MLP had <b>2 matrices</b> of size (dim × 4·dim).
- Total parameters = 2 · (dim · 4·dim) = <b>8 · dim²</b>.

<br>
In <b>SwiGLU</b>, there are <b>3 matrices</b> (w_gate, w_up, w_down):
- If we used 4·dim, parameters would be 3 · (dim · 4·dim) = 12 · dim² (a 50% compute increase!).
- To keep compute and parameters <b>identical</b> to the standard 2-matrix MLP, we solve:
<pre style='font-size: 16px; background: #161b22; color: #d29922; padding: 12px; border-radius: 8px;'>
3 · (dim · hidden_dim) = 8 · dim²
hidden_dim = (8 / 3) · dim ≈ 2.67 · dim
</pre>

### Example with dim = 768:
- Raw hidden dim: (8 / 3) · 768 = <b>2048</b>
- 2048 is cleanly divisible by 256 for peak GPU Tensor Core alignment!

In [ ]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.model.mlp import SwiGLUFFN

# Instantiate layer with dim=768
dim = 768
ffn = SwiGLUFFN(dim=dim)

# Count parameters in our 3 SwiGLU matrices:
n_params = sum(p.numel() for p in ffn.parameters())
print(f"Model Dimension:        {dim}")
print(f"SwiGLU Hidden Dim:      {ffn.w_gate.out_features} (Exactly 8/3 of 768!)")
print(f"Total Parameters:       {n_params:,}")

# Pass a batch of 2 tokens through the layer
x = torch.randn(1, 2, dim)
out = ffn(x)
print(f"Input shape:            {list(x.shape)}")
print(f"Output shape:           {list(out.shape)}")
assert out.shape == x.shape
print("\nSUCCESS: SwiGLU forward pass fully verified!")